# 05 Train the random forest

**What a random forest is.** A decision tree is a list of yes/no questions about the 14 numbers, like "is ring_0
above 0.3?", ending in an answer. One tree on its own learns its training data too well and makes odd mistakes. A
random forest trains a few hundred trees, each on a slightly different random part of the data, and lets them
vote. The mistakes of one tree get outvoted by the others.

**Why this model.** A forest is a strong, fast starting point before anything deep. It
trains in seconds on my laptop, answers fast, shows which features it used, and can give 8 answers at once, one
per defect (FR-01).

**What this notebook does.** It trains the forest on the training part, checks whether it learned or memorised,
tries six settings on the validation part, keeps the best, shows which features it used, times it, and saves it. The
test part isn't touched here.

**In:** `output/features.npz`, `output/split.npz`, `output/baseline_validation.csv` **Out:** `output/forest.joblib`, `output/timings.csv`, `output/settings_tried.csv`

## 1. Load the features

**Why this step:** The forest learns from the table notebook 03 made, not from the maps.

**The problem:** Load the training and validation features and their answers.

**How:** `allow_pickle=True` is needed because the file also holds the feature names as text.

In [1]:
import csv
import os
import time

import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier

import wafer

f = np.load("output/features.npz", allow_pickle=True)
X_train = f["X_train"]
y_train = f["y_train"]
X_val = f["X_val"]
y_val = f["y_val"]

print("Training on", X_train.shape[0], "wafers with", X_train.shape[1], "features each")
print("Every answer has", y_train.shape[1], "positions")

Training on 25822 wafers with 14 features each
Every answer has 8 positions


**What you should see:** 25,822 wafers with 14 features, and 8 positions per answer.

## 2. Train it

**Why this step:** This is the actual learning: the trees work out which questions about the 14 numbers split the defects best.

**The problem:** Train a forest that gives all 8 answers at once.

**How:** `RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)` makes the forest. `n_estimators` is how many trees vote. `random_state=42` is the seed again: the same forest every run (QC-06). `n_jobs=-1` uses every core of the processor, because the trees don't depend on each other and can be built at the same time.\
`forest.fit(X_train, y_train)` is the training. Because `y_train` has 8 columns, one forest learns all 8 defects at once. That's called multi-label.

In [2]:
forest = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

start = time.perf_counter()
forest.fit(X_train, y_train)
training_seconds = time.perf_counter() - start

print(f"Trained in {training_seconds:.1f} s (QC-07 allows 60 minutes)")

Trained in 2.2 s (QC-07 allows 60 minutes)


**What you should see:** The training time, next to the hour QC-07 allows.

## 3. Learned, or memorised?

**Why this step:** A model that memorises its training data looks perfect on it and fails on new wafers. That's called overfitting. The only way to see it is to score a part the model hasn't trained on.

**The problem:** Compare the score on the training part with the score on the validation part.

**How:** `forest.predict(X)` gives the forest's answers, 8 columns of 0 or 1. Then the same scores as in notebook 04: recall per defect, and all 8 right. A small gap between training and validation is normal. A big gap means memorising.

In [3]:
train_recall, _ = wafer.scores_per_defect(y_train, forest.predict(X_train))
val_recall, _ = wafer.scores_per_defect(y_val, forest.predict(X_val))

for i in range(len(wafer.DEFECTS)):
    gap = (train_recall[i] - val_recall[i]) * 100
    print(f"{wafer.DEFECTS[i]}: training {train_recall[i] * 100:.1f}%, validation {val_recall[i] * 100:.1f}%, gap {gap:.1f}")

train_all = wafer.share_all_eight_right(y_train, forest.predict(X_train))
val_all = wafer.share_all_eight_right(y_val, forest.predict(X_val))
print()
print(f"All 8 right, training: {train_all * 100:.1f}%")
print(f"All 8 right, validation: {val_all * 100:.1f}%")

Center: training 100.0%, validation 93.7%, gap 6.3
Donut: training 100.0%, validation 93.4%, gap 6.6
Edge-Loc: training 100.0%, validation 84.9%, gap 15.1
Edge-Ring: training 100.0%, validation 96.2%, gap 3.8
Loc: training 100.0%, validation 88.5%, gap 11.5
Near-Full: training 100.0%, validation 100.0%, gap 0.0
Scratch: training 100.0%, validation 85.0%, gap 15.0
Random: training 100.0%, validation 68.1%, gap 31.9



All 8 right, training: 100.0%
All 8 right, validation: 68.1%


**What you should see:** Recall per defect on training and on validation, the gap between them, and all 8 right on
both. Fully grown trees almost always get their own training wafers right, so a perfect training score on its own
doesn't prove memorising. The gap to validation is what to look at: for which defect is it biggest? Step 4 tests
whether holding the trees back closes it.

## 4. Try six settings

**Why this step:** Settings like the number of trees are my choice, not something the forest learns. I pick them on validation, so the test part stays a clean final exam.

**The problem:** Try more and fewer trees, two ways to hold the trees back, and one way to give the rare defects more weight, and see what helps.

**How:** `try_forest(model, name)` is a small function of mine: it trains one model, times it, scores it on validation, prints three numbers and remembers them, the trained model included, so step 5 doesn't have to train it again. I made it a function so I don't copy the same lines six times.Two ways to hold a tree back from memorising: `max_depth=20` stops a tree after 20 questions, and `min_samples_leaf=5` means every final answer of a tree must be based on at least 5 training wafers.`class_weight="balanced"` is for the rare defects. Random and Near-Full have far fewer training wafers than the others, so a tree learns little from them. With this setting a wafer of a rare defect counts as heavier, as if it were there more often.I look at three numbers: all 8 right; the lowest recall of the 8 defects, because QC-01 asks for at least 70% on *every* defect; and the lowest F1 of the 8, because F1 only goes up when recall and precision both do (notebook 04).Under the six settings I print the baseline from notebook 04 on the same validation wafers, read from `output/baseline_validation.csv`. Every setting should be far above it.

In [4]:
tried = []    # every setting I try, with its scores and the trained model


def try_forest(model, name):
    start = time.perf_counter()
    model.fit(X_train, y_train)
    seconds = time.perf_counter() - start

    predicted = model.predict(X_val)
    recall, precision = wafer.scores_per_defect(y_val, predicted)
    f1 = wafer.f1_per_defect(recall, precision)
    all_right = wafer.share_all_eight_right(y_val, predicted)

    tried.append({"setting": name, "trees": model.n_estimators, "all_right": all_right,
                  "lowest_recall": recall.min(), "lowest_f1": f1.min(), "seconds": seconds, "model": model})
    print(f"{name}: all 8 right {all_right * 100:.1f}%, lowest recall {recall.min() * 100:.1f}%, "
          f"lowest F1 {f1.min():.2f}, {seconds:.1f} s")


try_forest(RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1), "100 trees")
try_forest(RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1), "300 trees")
try_forest(RandomForestClassifier(n_estimators=600, random_state=42, n_jobs=-1), "600 trees")
try_forest(RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1),
           "300 trees, min_samples_leaf=5")
try_forest(RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42, n_jobs=-1),
           "300 trees, max_depth=20")
try_forest(RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1),
           "300 trees, class_weight=balanced")

# The baseline from notebook 04 on the same validation wafers, to compare with
baseline_all_right = None
with open("output/baseline_validation.csv", encoding="utf-8") as file:
    for row in csv.DictReader(file):
        if row["baseline"] == "Simple model (baseline)" and row["defect"] == "all":
            baseline_all_right = float(row["value"])
print(f"Baseline (notebook 04): all 8 right {baseline_all_right * 100:.1f}%")

# Saved, so notebook 09 can draw the score against the number of trees
os.makedirs("output", exist_ok=True)
with open("output/settings_tried.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["setting", "trees", "all_eight_right_val", "lowest_recall_val", "lowest_f1_val", "seconds"])
    for t in tried:
        writer.writerow([t["setting"], t["trees"], round(t["all_right"], 4), round(t["lowest_recall"], 4),
                         round(t["lowest_f1"], 4), round(t["seconds"], 1)])

100 trees: all 8 right 68.0%, lowest recall 69.0%, lowest F1 0.81, 0.7 s


300 trees: all 8 right 68.1%, lowest recall 68.1%, lowest F1 0.81, 2.0 s


600 trees: all 8 right 68.4%, lowest recall 68.1%, lowest F1 0.81, 4.4 s


300 trees, min_samples_leaf=5: all 8 right 66.5%, lowest recall 66.4%, lowest F1 0.80, 1.9 s


300 trees, max_depth=20: all 8 right 68.2%, lowest recall 69.0%, lowest F1 0.81, 2.2 s


300 trees, class_weight=balanced: all 8 right 66.9%, lowest recall 74.3%, lowest F1 0.81, 2.1 s
Baseline (notebook 04): all 8 right 31.1%


**What you should see:** Six settings, each with all 8 right, the lowest recall, the lowest F1 and the time, and
the baseline under them. Do more trees help, or only cost time? Does holding the trees back help? Does giving the
rare defects more weight lift the lowest recall? If no setting moves the score much, the forest isn't the limit,
and the 14 features it gets to see are the more likely one.

## 5. Keep the best one

**Why this step:** One model goes forward to the final exam. This is the decision that counts.

**The problem:** Pick the setting from step 4 that does best on its weakest defect.

**How:** I go through the settings and keep the one with the highest lowest F1. The weakest defect decides, because QC-01 asks for every defect, not the average. F1 instead of recall, so a setting can't win by saying yes more often. If two settings score the same, the first one in the list stays, and the list starts with the fastest. The model was already trained in step 4, so I take it from there, with its training time for QC-07.

In [5]:
best = tried[0]
for t in tried:
    if t["lowest_f1"] > best["lowest_f1"]:
        best = t

forest = best["model"]
training_seconds = best["seconds"]
print(f"Kept: {best['setting']} (lowest F1 {best['lowest_f1']:.2f}, trained in {training_seconds:.1f} s)")

Kept: 300 trees, max_depth=20 (lowest F1 0.81, trained in 2.2 s)


**What you should see:** Which setting won, its lowest F1, and how long it took to train.

## 6. Which features did it use?

**Why this step:** This is how I can explain *why* the model names a pattern, and it shows whether my feature design worked.

**The problem:** See how much the trees used each of the 14 features.

**How:** `forest.feature_importances_` gives one number per feature: how much the trees used it to split the wafers. They add up to 1. `np.argsort(...)[::-1]` puts the biggest first (`[::-1]` flips the order).

In [6]:
importance = forest.feature_importances_
order = np.argsort(importance)[::-1]

for i in order:
    print(f"{wafer.FEATURE_NAMES[i]}: {importance[i]:.3f}")

ring_4: 0.200
ring_3: 0.146
ring_0: 0.118
ring_2: 0.095
ring_1: 0.088
share_failed: 0.084
slice_6: 0.040
slice_5: 0.037
slice_3: 0.034
slice_0: 0.033
slice_1: 0.032
slice_2: 0.031
slice_7: 0.031
slice_4: 0.031


**What you should see:** The 14 features, most used first. Look at where the rings end up and where the slices end
up. I designed the slices to tell Edge-Loc from Edge-Ring: does this list say they did much?

## 7. How fast is it? (QC-08 and QC-09)

**Why this step:** In a factory a wafer has to be checked as it comes off the line. QC-08 asks for under 50 ms per wafer, QC-09 for 1000 wafers in under 10 seconds.

**The problem:** Time one wafer from map to answer, and a batch of 1000.

**How:** I time the whole way, making the features included, because a factory would have to do that too. Timing only the forest would make it look faster than it is. One wafer is the average of 50 runs, so one slow run doesn't decide it. A batch is faster per wafer, because the forest handles them all in one go.\
**One core for one wafer:** `n_jobs=-1` starts a worker on every core for every prediction. For a single row starting them costs more than the prediction itself, so for one wafer I switch to one core with `forest.set_params(n_jobs=1)`, and back to every core for the batch.

In [7]:
split = np.load("output/split.npz")
maps = split["maps"]
val_idx = split["val_idx"]

# QC-08: one wafer at a time, features included. One core, starting the workers costs more than one row
forest.set_params(n_jobs=1)
times = []
for i in val_idx[:50]:
    start = time.perf_counter()
    x = np.array([wafer.features(maps[i])])
    forest.predict(x)
    times.append((time.perf_counter() - start) * 1000)
one_wafer_ms = np.mean(times)
print(f"QC-08, one wafer: {one_wafer_ms:.1f} ms (allowed 50 ms)")

# QC-09: 1000 wafers at once, on every core again
forest.set_params(n_jobs=-1)
start = time.perf_counter()
forest.predict(wafer.feature_table(maps[val_idx[:1000]]))
batch_seconds = time.perf_counter() - start
print(f"QC-09, 1000 wafers: {batch_seconds:.2f} s (allowed 10 s)")

QC-08, one wafer: 8.6 ms (allowed 50 ms)
QC-09, 1000 wafers: 0.20 s (allowed 10 s)


**What you should see:** The time for one wafer next to QC-08's 50 ms, and for 1000 wafers next to QC-09's 10 s.
Times differ a bit between runs, depending on what else the laptop is doing.

## 8. Save the model and the times

**Why this step:** Notebooks 06, 08 and 09 load the trained forest instead of training again. In real use nobody retrains before checking a wafer either.

**The problem:** Save the forest to a file, and the three times to a csv.

**How:** `joblib.dump(forest, path, compress=3)` saves the trained forest. A forest stores every question of every tree, so it gets big, tens of MB. `compress=3` packs it, but 300 full trees stay big. `output/` is not in git, because a trained model is a result, not code.

In [8]:
os.makedirs("output", exist_ok=True)
joblib.dump(forest, "output/forest.joblib", compress=3)
print(f"Saved output/forest.joblib ({os.path.getsize('output/forest.joblib') / 1e6:.1f} MB)")

with open("output/timings.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["criterion", "measured", "target"])
    writer.writerow(["QC-07 training (s)", round(training_seconds, 1), 3600])
    writer.writerow(["QC-08 one wafer (ms)", round(one_wafer_ms, 1), 50])
    writer.writerow(["QC-09 1000 wafers (s)", round(batch_seconds, 2), 10])
print("Saved output/timings.csv")

Saved output/forest.joblib (84.9 MB)
Saved output/timings.csv


**What you should see:** Two "Saved" lines, the model with its size.

## What this means for the next notebooks

The forest is trained and saved, with the setting step 5 kept. How well it does on validation next to the baseline, and which
features carry it, is in steps 3, 4 and 6. Notebook 06 opens the test part to give the real score.

## Functions in this notebook

| Function | What it does | Where to read more |
|---|---|---|
| `RandomForestClassifier` | the model: a few hundred decision trees that vote | [docs](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) |
| `.fit` | trains the model on the inputs and the answers | [docs](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) |
| `.predict` | gives the model's answers, 0 or 1 per defect | [docs](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) |
| `.feature_importances_` | how much the trees used each feature | [docs](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) |
| `time.perf_counter` | a clock in seconds, for timing | [docs](https://docs.python.org/3/library/time.html#time.perf_counter) |
| `joblib.dump` | saves the trained model to a file | [docs](https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html) |
| `csv.DictReader` | reads a .csv row by row as name to value | [docs](https://docs.python.org/3/library/csv.html#csv.DictReader) |
| `csv.writer` | writes rows to a .csv file | [docs](https://docs.python.org/3/library/csv.html) |